# PostgreSQL MVCC Internals, Tuple Versioning & Snapshot Isolation

Interactive hands-on sandbox exploring internal storage mechanics, algorithmic data structures, and architectural invariants.


In [ ]:
import sys
from pathlib import Path

# Prepend project_solution to sys.path so Track A internal engines import cleanly
ps_dir = Path('.').resolve() / 'project_solution'
if str(ps_dir) not in sys.path:
    sys.path.insert(0, str(ps_dir))

print(f'Python runtime: {sys.version.split()[0]}')
print('Loaded internal mechanics for: Module_05_PostgreSQL_MVCC_Indexing_EXPLAIN')


## 1. Engine Initialization & Setup

Importing the module's Track A internal simulation engine and instantiating state.


In [ ]:
from mvcc_engine import MVCCTable, Snapshot

# Initialize in-memory MVCC Table
mvcc = MVCCTable("accounts")

# Transaction 100 inserts initial rows
mvcc.insert("acc_1", {"name": "Alice", "balance": 1000}, xid=100)
mvcc.insert("acc_2", {"name": "Bob", "balance": 2500}, xid=100)
print(f"Tx 100 inserted 2 rows. Physical heap tuple count: {len(mvcc.tuples)}")


## 2. Core Architectural Operations & State Mutation

Executing data mutations, transactions, or indexing procedures.


In [ ]:
# Transaction 110 updates Alice's balance (1000 -> 1200)
# In MVCC, UPDATE does NOT overwrite; it appends a new tuple version and sets xmax on the old tuple!
mvcc.update("acc_1", {"name": "Alice", "balance": 1200}, xid=110)

print(f"Heap tuple count after update: {len(mvcc.tuples)} (Old version preserved in-place!)")
for idx, tup in enumerate(mvcc.tuples):
    print(f"  Tuple {idx}: xmin={tup.xmin}, xmax={tup.xmax}, data={tup.data}")


## 3. Performance Micro-Benchmarking & Invariant Verification

Evaluating execution latency, cache hits, or computational trade-offs.


In [ ]:
# Snapshot Isolation Visibility:
# Snapshot taken before Tx 110 commits sees Alice's balance = 1000
snap_old = Snapshot(snapshot_xid=105, xmin=100, xmax=106, active_xids=set())
visible_old = mvcc.select(snap_old)

# Snapshot taken after Tx 110 commits sees Alice's balance = 1200
snap_new = Snapshot(snapshot_xid=115, xmin=100, xmax=116, active_xids=set())
visible_new = mvcc.select(snap_new)

print("Data visible to Snapshot OLD (pre-update):", [(r['name'], r['balance']) for r in visible_old])
print("Data visible to Snapshot NEW (post-update):", [(r['name'], r['balance']) for r in visible_new])


## 4. Architectural Invariant Verification

Asserting mathematical correctness and durability invariants.


In [ ]:
# Invariant check: Readers never block writers, writers never block readers
assert any(r['balance'] == 1000 for r in visible_old), "Old snapshot must see pre-update value"
assert any(r['balance'] == 1200 for r in visible_new), "New snapshot must see committed update"
assert len(mvcc.tuples) == 3, "MVCC heap must contain both dead and live tuple versions until VACUUM"
print("[+] MVCC HeapTuple and Snapshot Isolation invariants verified successfully!")


## Summary & Operational DBRE Best Practices

1. **Never bypass serialization contracts:** Always enforce binary-safe schemas and validated boundaries.
2. **Monitor buffer and memory allocations:** Understand the latency cliff when in-memory structures spill to disk.
3. **Ensure idempotency across replication tiers:** Distributed operations must survive retries without corrupting state.
